# 03 — Training

Fine-tune encoder models and run LLM baselines.  
All runs use the same hyperparameters — the only variable is which density column is used for weighting.

**Sections**
1. Configuration
2. Define experiments (dataset × density column)
3. Fine-tuning runs
4. LLM baselines
5. Results summary

In [1]:
import sys, os
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

import json
import yaml
import pandas as pd

from src.training import TrainingConfig, train, run_baselines

c:\Users\Alexandre\miniconda3\envs\faiss2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

Edit `TrainingConfig` fields below to change model, hyperparameters, or output paths.  
All experiments in this notebook share the same config — only `density_column` varies per run.

In [2]:
CONFIG_PATH = "configs/datasets.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

EMBEDDINGS_ROOT   = cfg["embedding"]["output_root"]
PREPROCESSED_ROOT = cfg["preprocessing"]["output_root"]
MODEL_NAME        = cfg["embedding"]["models"][0]
MODEL_SLUG        = MODEL_NAME.replace("/", "_")
K_VALUES          = cfg["embedding"]["k_values"]

# Russian is reference-only: not used for fine-tuning, only for density + eval
TRAIN_DATASETS    = cfg.get("train_datasets", ["toxigen"])

train_cfg = cfg["training"]

base_config = TrainingConfig(
    model_id     = train_cfg["models"][0],
    batch_size   = train_cfg["batch_size"],
    learning_rate= train_cfg["learning_rate"],
    num_epochs   = train_cfg["epochs"],
    max_length   = train_cfg["max_length"],
    random_state = train_cfg["random_state"],
    output_root  = train_cfg["output_root"],
)

print("Model          :", base_config.model_id)
print("Epochs         :", base_config.num_epochs)
print("LR             :", base_config.learning_rate)
print("Train datasets :", TRAIN_DATASETS)
print("Output         :", base_config.output_root)

Model          : answerdotai/ModernBERT-base
Epochs         : 1
LR             : 2e-5
Train datasets : ['toxigen']
Output         : outputs/3_training


## 2. Define Experiments

Each experiment is `(dataset, density_column)`.  
- `density_column = None` → train without weighting (baseline encoder)
- `density_column = 'density_k5_ratio'` → weight by Russian/All ratio at K=5 on raw embeddings
- `density_column = 'density_pca_k5_ratio'` → same but PCA space

Edit the list below to add/remove experiments.

In [3]:
# Build experiment list: one fine-tune per (train_dataset × density_column)
# Russian is excluded — it is the reference group used for density, not for training.
density_columns = [None]  # baseline: no weighting
for k in K_VALUES:
    density_columns.append(f"density_k{k}_ratio")       # raw space
    density_columns.append(f"density_pca_k{k}_ratio")   # PCA space

experiments = [
    {"dataset": ds, "density_column": dc}
    for ds in TRAIN_DATASETS
    for dc in density_columns
]

print(f"{len(experiments)} fine-tunes planned:")
for ex in experiments:
    tag = f"{ex['dataset']}__{ex['density_column'] or 'no_density'}"
    print(f"  {tag}")

7 fine-tunes planned:
  toxigen__no_density
  toxigen__density_k5_ratio
  toxigen__density_pca_k5_ratio
  toxigen__density_k100_ratio
  toxigen__density_pca_k100_ratio
  toxigen__density_k1000_ratio
  toxigen__density_pca_k1000_ratio


## 3. Fine-Tuning Runs

Each run loads `outputs/2_embeddings/{dataset}/{model_slug}/densities.csv`,  
fine-tunes the model, and saves the checkpoint + `metrics.json` to `outputs/3_training/`.

In [4]:
all_metrics = {}

for ex in experiments:
    ds = ex["dataset"]
    dc = ex["density_column"]
    dataset_tag = ds

    density_csv = os.path.join(EMBEDDINGS_ROOT, ds, MODEL_SLUG, "densities.csv")
    if not os.path.exists(density_csv):
        print(f"[SKIP] {density_csv} not found")
        continue

    # Clone config and set density column for this run
    from dataclasses import replace
    run_config = TrainingConfig(
        model_id      = base_config.model_id,
        batch_size    = base_config.batch_size,
        learning_rate = base_config.learning_rate,
        num_epochs    = base_config.num_epochs,
        max_length    = base_config.max_length,
        random_state  = base_config.random_state,
        output_root   = base_config.output_root,
        density_column= dc,
    )

    run_name = run_config.run_name(dataset_tag)
    metrics_path = os.path.join(run_config.output_dir(dataset_tag), "metrics.json")

    if os.path.exists(metrics_path):
        print(f"[CACHE] {run_name} — loading existing metrics")
        with open(metrics_path) as f:
            all_metrics[run_name] = json.load(f)
        continue

    print(f"\n{'='*60}")
    print(f"Training: {run_name}")
    print(f"{'='*60}")
    metrics = train(density_csv=density_csv, dataset_tag=dataset_tag, config=run_config)
    all_metrics[run_name] = metrics

print("\nAll fine-tuning runs complete.")


Training: toxigen__no_density


Loading weights: 100%|██████████| 136/136 [00:00<00:00, 5150.94it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Step,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1,Auc Roc
1000,0.263072,0.250499,0.896952,0.852351,0.831801,0.758963,0.793714,0.953181
2000,0.278624,0.209473,0.911715,0.858645,0.897354,0.747521,0.815613,0.967108
3000,0.224967,0.227312,0.910420,0.909000,0.784441,0.906026,0.840861,0.970376
4000,0.217668,0.207895,0.914186,0.913792,0.790816,0.912967,0.847513,0.973780
5000,0.197934,0.192432,0.922574,0.910889,0.829006,0.886423,0.856753,0.975208
6000,0.197635,0.183106,0.922833,0.915897,0.820796,0.901373,0.859199,0.976865
7000,0.158144,0.198028,0.923829,0.915363,0.825883,0.897635,0.860265,0.976738
8000,0.202919,0.186747,0.926440,0.885695,0.907071,0.800381,0.850393,0.977797
9000,0.207996,0.177146,0.928352,0.886250,0.916842,0.798093,0.853356,0.979918
10000,0.159070,0.163033,0.933313,0.907950,0.885859,0.854844,0.870075,0.980451


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.53s/it]


Training Loss,Validation Loss,Step,Accuracy,Balanced Accuracy,Precision,Recall,F1,Auc Roc
0.155950,0.157659,12548,0.935246,0.914978,0.878706,0.872540,0.875612,0.981786


...

Step,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1,Auc Roc
1000,0.427062,0.923798,0.792567,0.806793,0.481191,0.830099,0.609226,0.878010
2000,0.520499,0.372004,0.889793,0.768599,0.807583,0.570056,0.668343,0.923207
3000,0.434471,0.442141,0.897818,0.769085,0.870881,0.558190,0.680326,0.928257
4000,0.728074,0.469103,0.875224,0.689705,0.936071,0.385785,0.546386,0.932428
5000,0.241624,0.427726,0.899464,0.863775,0.714728,0.805307,0.757319,0.950254
6000,0.366667,0.397523,0.910770,0.848426,0.785023,0.746293,0.765168,0.948040
7000,0.467221,0.338972,0.908229,0.805427,0.854877,0.637014,0.730038,0.950438
8000,0.418568,0.350346,0.914448,0.854931,0.793907,0.757428,0.775239,0.955490
9000,0.455540,0.374259,0.910431,0.800845,0.884499,0.621317,0.729909,0.954835
10000,0.268965,0.358046,0.912444,0.808285,0.879783,0.637648,0.739397,0.956083


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.90s/it]


Training Loss,Validation Loss,Step,Accuracy,Balanced Accuracy,Precision,Recall,F1,Auc Roc
0.418772,0.302530,12548,0.918330,0.845552,0.833019,0.726325,0.776022,0.961997


Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.02s/it]



All fine-tuning runs complete.


## 4. LLM Baselines

Requires Ollama running locally (`ollama serve`).  
Runs zero-shot, context, and few-shot classification on the Russian test split.

In [5]:
RUN_BASELINES = False  # set True to run (requires Ollama)

if RUN_BASELINES:
    # Use the Russian annotated test set
    russian_test_csv = os.path.join(PREPROCESSED_ROOT, "russian", "test.csv")
    if os.path.exists(russian_test_csv):
        baseline_results = run_baselines(
            test_csv=russian_test_csv,
            dataset_tag="russian_annotated",
            config=base_config,
        )
        print("Baseline results:")
        for mode, metrics in baseline_results.items():
            print(f"  {mode}: F1={metrics['f1']:.4f}  Acc={metrics['accuracy']:.4f}")
    else:
        print(f"Russian test CSV not found at {russian_test_csv}")
else:
    print("Baselines skipped (RUN_BASELINES=False).")

Baselines skipped (RUN_BASELINES=False).


## 5. Results Summary

In [6]:
if all_metrics:
    rows = []
    for run_name, m in all_metrics.items():
        parts = run_name.split("__", 1)
        rows.append({
            "run": run_name,
            "dataset": parts[0],
            "density": parts[1] if len(parts) > 1 else "n/a",
            "f1":               m.get("f1", float("nan")),
            "accuracy":         m.get("accuracy", float("nan")),
            "balanced_accuracy": m.get("balanced_accuracy", float("nan")),
            "auc_roc":          m.get("auc_roc", float("nan")),
        })

    summary = pd.DataFrame(rows).sort_values("f1", ascending=False)
    display(summary.style.format({
        "f1": "{:.4f}",
        "accuracy": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "auc_roc": "{:.4f}",
    }))
else:
    print("No metrics collected yet — run the training cells first.")

,run,dataset,density,f1,accuracy,balanced_accuracy,auc_roc
3,toxigen__density_k100_ratio,toxigen,density_k100_ratio,0.8757,0.9353,0.9154,0.9817
1,toxigen__density_k5_ratio,toxigen,density_k5_ratio,0.8756,0.9352,0.9151,0.9818
0,toxigen__no_density,toxigen,no_density,0.8756,0.9352,0.9150,0.9818
5,toxigen__density_k1000_ratio,toxigen,density_k1000_ratio,0.8561,0.9354,0.9022,0.9790
6,toxigen__density_pca_k1000_ratio,toxigen,density_pca_k1000_ratio,0.7760,0.9183,0.8456,0.9620
4,toxigen__density_pca_k100_ratio,toxigen,density_pca_k100_ratio,0.7741,0.9222,0.8415,0.9588
2,toxigen__density_pca_k5_ratio,toxigen,density_pca_k5_ratio,0.7558,0.9121,0.8316,0.9586
